# Empirical Bernstein Mathematics: Theory and Formulas

This notebook explains the mathematical theory behind **Empirical Bernstein Stopping (EBS)**, based on Algorithm 2 from Gresch, Tepe, and Kliesch (arXiv:2502.01730v1).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qamp_shotplanner.planners.hoeffding import HoeffdingPlanner
from qamp_shotplanner.planners.empirical_bernstein import (
    eb_radius,
    eb_radius_modified,
    geom_checkpoints,
    ebs_delta_schedule,
)
from qamp_shotplanner.stats.running_stats import RunningStats

np.random.seed(42)

## 1. Review: Hoeffding's Inequality

For a bounded random variable $X \in [a, b]$ with mean $\mu = \mathbb{E}[X]$, Hoeffding's inequality states:

$$
\mathbb{P}\left( \left| \bar{X}_n - \mu \right| \geq \epsilon \right) \leq 2 \exp\left(-\frac{2n\epsilon^2}{R^2}\right)
$$

where $R = b - a$ and $\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i$.

### Shot Planning Formula

To achieve confidence $1 - \delta$, we set the right side equal to $\delta$ and solve for $n$:

$$
n = \frac{R^2}{2\epsilon^2} \ln\left(\frac{2}{\delta}\right)
$$

### Key Properties

- **Data-independent**: Uses only the range $R$, not the variance
- **Conservative**: Assumes worst-case variance $\sigma^2 = R^2/4$
- **Simple**: Easy to compute and understand

In [ ]:
# Hoeffding example
epsilon = 0.02
delta = 0.01
a, b = 0.0, 1.0

hoeffding = HoeffdingPlanner(epsilon_stat=epsilon, delta=delta, a=a, b=b)
n_hoeffding = hoeffding.planned_shots()

print("=== Hoeffding Planning ===")
print(f"Range R = b - a = {b - a}")
print(f"Tolerance ε = {epsilon}")
print(f"Confidence 1 - δ = {1 - delta:.2%}")
print(f"Planned shots: {n_hoeffding:,}")
print(f"\nImplicit assumption: σ² ≤ R²/4 = {(b-a)**2 / 4}")

## 2. Empirical Bernstein Inequality (Equation 4)

The **empirical Bernstein inequality** improves on Hoeffding by using the **observed variance**:

$$
\mathbb{P}\left( \left| \bar{X}_N - \mu \right| \geq \epsilon_N \right) \leq \delta
$$

where the **empirical Bernstein radius** is:

$$
\epsilon_N = \bar{\sigma}_N \sqrt{\frac{2\ln(3/\delta)}{N}} + R \frac{3\ln(3/\delta)}{N}
$$

### Two Terms Explained

1. **Variance-dependent term**: $\bar{\sigma}_N \sqrt{\frac{2\ln(3/\delta)}{N}}$
   - Uses the **biased empirical standard deviation** $\bar{\sigma}_N = \sqrt{\bar{\sigma}_N^2}$
   - Biased variance: $\bar{\sigma}_N^2 = \frac{1}{N}\sum_{i=1}^N (X_i - \bar{X}_N)^2$
   - **Dominates** when variance is significant

2. **Range-dependent term**: $R \frac{3\ln(3/\delta)}{N}$
   - Uses the full range $R = b - a$
   - **Correction term** that ensures validity even with small variance
   - Becomes negligible as $N$ grows

### Connection to Hoeffding

When $\bar{\sigma}_N^2 \approx R^2/4$ (worst-case), the EB radius approaches the Hoeffding bound (with slightly different constants).

In [ ]:
# Demonstrate EB radius computation
n = 1000
R = b - a

# Low variance case
var_low = 0.09
eps_eb_low = eb_radius(n=n, R=R, var_biased=var_low, delta=delta)

# Worst-case variance
var_worst = 0.25
eps_eb_worst = eb_radius(n=n, R=R, var_biased=var_worst, delta=delta)

# Hoeffding equivalent radius
eps_hoeffding = R / np.sqrt(2 * n / np.log(2 / delta))

print("=== EB Radius vs Hoeffding (n=1000) ===")
print(f"\nLow variance (σ²={var_low}):")
print(f"  EB radius: {eps_eb_low:.6f}")
print(f"  Hoeffding: {eps_hoeffding:.6f}")
print(f"  Ratio: {eps_eb_low / eps_hoeffding:.2f}x")

print(f"\nWorst-case variance (σ²={var_worst}):")
print(f"  EB radius: {eps_eb_worst:.6f}")
print(f"  Hoeffding: {eps_hoeffding:.6f}")
print(f"  Ratio: {eps_eb_worst / eps_hoeffding:.2f}x")

print(f"\nEB is tighter when variance is low!")

## 3. Modified EB Radius (Algorithm 2)

The paper introduces a **modified version** for use at checkpoints:

$$
\epsilon_n = \bar{\sigma}_n \sqrt{\frac{2x}{n}} + R \frac{x}{n}
$$

where:

$$
x = -\alpha \ln\left(\frac{\delta_k}{3}\right)
$$

### Parameters

- **$\delta_k$**: Per-checkpoint failure probability (allocated from total $\delta$)
- **$\alpha$**: Mid-interval tightness parameter (default: 1.0)
  - $\alpha < 1$: More conservative (larger radius, more shots)
  - $\alpha > 1$: More aggressive (tighter radius, fewer shots)
  - $\alpha = 1$: Standard choice

### Why Modify?

The modification allows for **per-checkpoint confidence allocation**, which is necessary when checking multiple times. The $\delta_k$ values must sum to $\delta$ to maintain overall confidence.

In [ ]:
# Compare modified EB radius with different alpha values
n = 1000
var_biased = 0.09
delta_k = 0.001  # Per-checkpoint delta

alphas = [0.5, 0.75, 1.0, 1.5, 2.0]
radii = []

print("=== Modified EB Radius: Alpha Sensitivity ===")
print(f"n = {n}, σ² = {var_biased}, δ_k = {delta_k}\n")

for alpha in alphas:
    eps = eb_radius_modified(n=n, R=R, var_biased=var_biased, delta_k=delta_k, alpha=alpha)
    radii.append(eps)
    print(f"α = {alpha:.2f}: ε_n = {eps:.6f}")

print(f"\nAlpha controls tightness: smaller α → larger radius (more conservative)")

## 4. Geometric Checkpoints

Instead of checking at every sample, EBS uses **geometric checkpoints**:

$$
n_k = \left\lceil \beta^k \right\rceil, \quad k = k_0, k_0+1, \ldots
$$

where:
- $\beta > 1$ is the growth factor (default: 1.1)
- $k_0 = \left\lceil \log_\beta(n_{\text{min}}) \right\rceil$ ensures first checkpoint $\geq n_{\text{min}}$
- Sequence continues until reaching $n_{\text{max}}$ (Hoeffding cap)

### Why Geometric?

1. **Efficiency**: Fewer checks than linear spacing
2. **Finite horizon**: Guarantees finite number of checks $K$
3. **Early sensitivity**: Dense checks early (when uncertainty is high)
4. **Late efficiency**: Sparse checks late (when uncertainty is low)

In [ ]:
# Generate geometric checkpoints
beta = 1.1
n_min = 10
n_max = n_hoeffding

checkpoints = geom_checkpoints(beta=beta, n_min=n_min, n_max=n_max)

print(f"=== Geometric Checkpoints ===")
print(f"β = {beta}, n_min = {n_min}, n_max = {n_max:,}\n")
print(f"Total checkpoints: {len(checkpoints)}")
print(f"First 10: {checkpoints[:10]}")
print(f"Last 10: {checkpoints[-10:]}")

# Compute spacing between checkpoints
spacings = np.diff(checkpoints)
print(f"\nSpacing statistics:")
print(f"  Min spacing: {np.min(spacings)}")
print(f"  Max spacing: {np.max(spacings)}")
print(f"  Mean spacing: {np.mean(spacings):.1f}")

In [ ]:
# Visualize checkpoint distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Checkpoints on log scale
ax1.plot(checkpoints, 'o-', markersize=4)
ax1.set_xlabel('Checkpoint index k', fontsize=11)
ax1.set_ylabel('Sample count n_k', fontsize=11)
ax1.set_title('Geometric Checkpoints (log scale)', fontsize=12, fontweight='bold')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# Plot 2: Checkpoint spacing
ax2.plot(spacings, 'o-', markersize=4, color='orange')
ax2.set_xlabel('Checkpoint index k', fontsize=11)
ax2.set_ylabel('Spacing Δn_k = n_{k+1} - n_k', fontsize=11)
ax2.set_title('Checkpoint Spacing (grows geometrically)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Geometric spacing avoids wasteful checking while ensuring finite K")

## 5. Delta Allocation: Uniform Schedule

With $K$ checkpoints, we need to allocate the total failure probability $\delta$ across all checks.

The paper recommends **uniform allocation**:

$$
\delta_k = \frac{\delta}{K}, \quad k = 1, \ldots, K
$$

This ensures:
$$
\sum_{k=1}^K \delta_k = \delta
$$

### Union Bound

By the union bound, the probability that **any** checkpoint violates its bound is at most:
$$
\mathbb{P}(\text{any violation}) \leq \sum_{k=1}^K \delta_k = \delta
$$

In [ ]:
# Compute delta schedule
K = len(checkpoints)
deltas = ebs_delta_schedule(delta=delta, K=K)

print(f"=== Delta Allocation ===")
print(f"Total δ = {delta}")
print(f"Number of checkpoints K = {K}")
print(f"Per-checkpoint δ_k = {deltas[0]:.6f}")
print(f"\nVerification: Σ δ_k = {sum(deltas):.6f} (should equal {delta})")
print(f"All deltas equal: {all(d == deltas[0] for d in deltas)}")

## 6. Running Statistics: Welford's Algorithm

To compute the empirical variance **online** (without storing all samples), we use **Welford's algorithm**:

### Update Equations

For each new sample $x_n$:

1. **Update mean**:
   $$
   \delta_n = x_n - \bar{X}_{n-1}
   $$
   $$
   \bar{X}_n = \bar{X}_{n-1} + \frac{\delta_n}{n}
   $$

2. **Update sum of squared differences** ($M_2$):
   $$
   M_{2,n} = M_{2,n-1} + \delta_n \cdot (x_n - \bar{X}_n)
   $$

3. **Compute biased variance**:
   $$
   \bar{\sigma}_n^2 = \frac{M_{2,n}}{n}
   $$

### Advantages

- **Memory efficient**: Only stores mean and $M_2$ (not all samples)
- **Numerically stable**: Avoids catastrophic cancellation
- **Single-pass**: Computes variance in one pass through data

In [ ]:
# Demonstrate Welford's algorithm
samples = np.random.binomial(1, 0.7, size=100)

# Using RunningStats
stats = RunningStats()
for x in samples:
    stats.update(x)

# Ground truth (NumPy)
mean_numpy = np.mean(samples)
var_biased_numpy = np.var(samples, ddof=0)  # Biased variance

print("=== Welford's Algorithm Validation ===")
print(f"Samples: n = {len(samples)}\n")

print("Mean:")
print(f"  RunningStats: {stats.mean:.6f}")
print(f"  NumPy:        {mean_numpy:.6f}")
print(f"  Match: {np.isclose(stats.mean, mean_numpy)}")

print("\nBiased variance:")
print(f"  RunningStats: {stats.variance_biased:.6f}")
print(f"  NumPy:        {var_biased_numpy:.6f}")
print(f"  Match: {np.isclose(stats.variance_biased, var_biased_numpy)}")

print("\nMemory usage: O(1) vs O(n) for storing all samples")

## 7. Stopping Criterion

At each checkpoint $k$ with $n_k$ samples, we:

1. **Compute empirical variance** $\bar{\sigma}_{n_k}^2$ using Welford's algorithm
2. **Compute modified EB radius**:
   $$
   \epsilon_{n_k} = \bar{\sigma}_{n_k} \sqrt{\frac{2x_k}{n_k}} + R \frac{x_k}{n_k}
   $$
   where $x_k = -\alpha \ln(\delta_k/3)$
3. **Check stopping criterion**:
   $$
   \text{if } \epsilon_{n_k} < \epsilon_{\text{stat}}, \text{ then STOP}
   $$
4. **Otherwise**: Continue to next checkpoint
5. **Safety**: If $n_k = n_{\text{max}}$ (Hoeffding cap), stop regardless

In [ ]:
# Simulate EBS stopping criterion
np.random.seed(42)
true_p = 0.8  # Biased coin
epsilon_target = 0.02

# Generate all samples upfront (for visualization)
all_samples = np.random.binomial(1, true_p, size=n_max)

# Track radius evolution
stats = RunningStats()
checkpoint_data = []

for k, n_k in enumerate(checkpoints):
    # Update stats up to this checkpoint
    prev_n = stats.n
    for i in range(prev_n, n_k):
        stats.update(float(all_samples[i]))
    
    # Compute radius
    delta_k = deltas[k]
    eps_n = eb_radius_modified(
        n=stats.n,
        R=R,
        var_biased=stats.variance_biased,
        delta_k=delta_k,
        alpha=1.0,
    )
    
    checkpoint_data.append({
        'k': k,
        'n': n_k,
        'mean': stats.mean,
        'var': stats.variance_biased,
        'radius': eps_n,
        'stopped': eps_n < epsilon_target,
    })
    
    # Would we stop here?
    if eps_n < epsilon_target:
        stop_index = k
        break
else:
    stop_index = len(checkpoints) - 1

print(f"=== EBS Stopping Simulation ===")
print(f"True p = {true_p}")
print(f"Target radius ε_stat = {epsilon_target}")
print(f"\nStopped at checkpoint k = {stop_index}")
print(f"Samples used: {checkpoint_data[stop_index]['n']:,}")
print(f"Final radius: {checkpoint_data[stop_index]['radius']:.6f}")
print(f"Final estimate: {checkpoint_data[stop_index]['mean']:.4f} (true: {true_p})")
print(f"\nShot savings: {n_max - checkpoint_data[stop_index]['n']:,} ({100*(1 - checkpoint_data[stop_index]['n']/n_max):.1f}%)")

In [ ]:
# Visualize radius evolution
import pandas as pd

df = pd.DataFrame(checkpoint_data)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Plot 1: Radius vs samples
ax1.plot(df['n'], df['radius'], 'o-', markersize=4, label='EB radius ε_n')
ax1.axhline(epsilon_target, color='red', linestyle='--', linewidth=2, label='Target ε_stat')
ax1.axvline(checkpoint_data[stop_index]['n'], color='green', linestyle=':', linewidth=2, label='Stopping point')
ax1.set_ylabel('Radius ε_n', fontsize=11)
ax1.set_title('EBS Radius Convergence', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log')

# Plot 2: Variance vs samples
ax2.plot(df['n'], df['var'], 'o-', markersize=4, color='purple', label='Empirical variance σ̄²_n')
ax2.axhline(true_p * (1 - true_p), color='orange', linestyle='--', linewidth=2, label='True variance')
ax2.axvline(checkpoint_data[stop_index]['n'], color='green', linestyle=':', linewidth=2, label='Stopping point')
ax2.set_xlabel('Number of samples n', fontsize=11)
ax2.set_ylabel('Variance σ̄²_n', fontsize=11)
ax2.set_title('Empirical Variance Convergence', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log')

plt.tight_layout()
plt.show()

print("\nKey observation: Radius decreases as more samples reveal the low variance")

## 8. Algorithm Summary: EBS Pseudocode

```
Algorithm: Empirical Bernstein Stopping (EBS)

Input:
  - ε_stat: Statistical error tolerance
  - δ: Total failure probability
  - a, b: Bounds of random variable (R = b - a)
  - β: Geometric checkpoint factor (default 1.1)
  - α: Mid-interval tightness (default 1.0)
  - n_min: Minimum samples before first check (default 10)

1. Compute n_max using Hoeffding: n_max = (R²/(2ε²)) ln(2/δ)

2. Generate geometric checkpoints:
   {n_k} = {⌈β^k⌉ : k = k_0, k_0+1, ...} until n_k > n_max
   where k_0 = ⌈log_β(n_min)⌉

3. Allocate delta uniformly: δ_k = δ / K for k = 1, ..., K

4. Initialize RunningStats (mean = 0, M2 = 0, n = 0)

5. For each checkpoint k:
     a. Sample from n_{k-1} to n_k, updating RunningStats
     b. Compute x_k = -α ln(δ_k / 3)
     c. Compute radius:
        ε_{n_k} = σ̄_{n_k} √(2x_k/n_k) + R (x_k/n_k)
     d. If ε_{n_k} < ε_stat:
          STOP and return (mean, n_k, ε_{n_k})
     e. If n_k == n_max:
          STOP (hit cap) and return (mean, n_max, ε_{n_max})

Output:
  - Estimate of mean
  - Number of samples used
  - Final radius
```

## 9. Comparison: EB Radius vs Hoeffding Radius

Let's directly compare the two confidence radii as a function of sample size:

In [ ]:
# Compare EB vs Hoeffding radius
n_range = np.logspace(1, 4.5, 50, dtype=int)
var_scenarios = [0.09, 0.16, 0.25]  # Low, medium, worst-case
var_labels = ['Low variance (σ²=0.09)', 'Medium variance (σ²=0.16)', 'Worst-case (σ²=0.25)']

fig, ax = plt.subplots(figsize=(12, 7))

colors = ['blue', 'purple', 'red']

for var, label, color in zip(var_scenarios, var_labels, colors):
    radii_eb = [eb_radius(n=n, R=R, var_biased=var, delta=delta) for n in n_range]
    ax.plot(n_range, radii_eb, '-', linewidth=2.5, color=color, label=f'EB: {label}')

# Hoeffding radius (data-independent)
radii_hoeff = [R / np.sqrt(2 * n / np.log(2 / delta)) for n in n_range]
ax.plot(n_range, radii_hoeff, '--', linewidth=2.5, color='black', label='Hoeffding (worst-case)')

# Target tolerance line
ax.axhline(epsilon, color='green', linestyle=':', linewidth=2, label=f'Target ε = {epsilon}')

ax.set_xlabel('Number of samples n', fontsize=12)
ax.set_ylabel('Confidence radius ε_n', fontsize=12)
ax.set_title('Empirical Bernstein vs Hoeffding: Radius Comparison', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("Key insights:")
print("  1. EB radius is always ≤ Hoeffding radius (for same n)")
print("  2. Low variance → EB reaches target ε much earlier")
print("  3. Worst-case variance → EB converges to Hoeffding")
print("  4. EB adapts to data, Hoeffding uses fixed worst-case")

## 10. Key Takeaways

### Mathematical Foundation

1. **Hoeffding**: Data-independent, uses worst-case variance $R^2/4$
   $$n = \frac{R^2}{2\epsilon^2} \ln\left(\frac{2}{\delta}\right)$$

2. **Empirical Bernstein**: Variance-aware, uses observed $\bar{\sigma}_n^2$
   $$\epsilon_n = \bar{\sigma}_n \sqrt{\frac{2\ln(3/\delta)}{n}} + R \frac{3\ln(3/\delta)}{n}$$

3. **Modified EB** (Algorithm 2): Per-checkpoint confidence allocation
   $$\epsilon_n = \bar{\sigma}_n \sqrt{\frac{2x}{n}} + R \frac{x}{n}, \quad x = -\alpha \ln(\delta_k/3)$$

### Key Components

- **Geometric checkpoints**: $n_k = \lceil \beta^k \rceil$ for efficient evaluation
- **Delta allocation**: Uniform $\delta_k = \delta / K$ ensures union bound
- **Welford's algorithm**: Online variance computation in O(1) memory
- **Stopping criterion**: $\epsilon_n < \epsilon_{\text{stat}}$ with Hoeffding cap

### Advantages

- Adapts to low variance (saves shots)
- Maintains safety (never exceeds Hoeffding cap)
- Memory efficient (online computation)
- Tunable (β, α parameters)

### Next Steps

In the following notebooks, we'll:
- Apply EBS to SWAP test fidelity estimation (Notebook 07)
- Compare EBS vs Hoeffding empirically (Notebook 08)
- Tune parameters β, α, n_min for optimal performance (Notebook 09)